# Stage 01a — Wizard Feed (T1 pre-labels + duplicate detection only)

Lightweight replacement for the *wizard-feed* responsibility of
`01_review.ipynb`. Produces `ml_predict_labels_<batch>.xlsx` for loading into
the HTML wizard (`UI_for_taxonomy_caracterization_15_3.html` (the current wizard; this notebook writes only T1 + duplicate rows, none of which changed v13->v15.3)), but skips the
expensive SigLIP T2/G1/M1/M2/M3 image classification entirely — this is a
fast **triage-only** pre-label pass so a human can approve/disapprove and
resolve duplicates *before* anyone pays for the full per-figure vision pass.

**Pipeline position:** `00b2_figure_crop_&_Brief_DD_matching` (crops figures,
matches them to description lines) → **`01a_wizard_feed`** (this notebook —
T1 SBERT pre-labels + SigLIP/SBERT duplicate detection) → human reviews T1
triage + duplicates in the HTML wizard → (a later, heavier stage — still
`01_review.ipynb`'s `run_stage01()`, or a future notebook — runs full T2/G1/
M1/M2/M3 SigLIP classification for the patents that survive triage).

**What this notebook writes to `ml_predict_labels_<batch>.xlsx`:**
- T1 metadata (title/abstract/assignee/pub_year/app_year/description of
  drawings) — always, since the reviewer needs it to triage at all.
- T1 pre-labels (scope/t1Field/t1Target) from PatentSBERTa — `Source=sbert`.
- Duplicate-image rows (`dupOfPatent`/`dupOfFig`) from SigLIP image-to-image
  comparison — only for figures actually flagged as a duplicate.
- Duplicate-patent rows (`isDuplicate`/`duplicateId`/`duplicateType`) from
  PatentSBERTa text-to-text comparison — only for patents actually flagged.

**What it does NOT write:** any T2/G1/M1/M2/M3 classification field. The HTML
wizard's `fresh()` state already defaults every one of those to null/false/
unset, so an absent row is functionally identical to "not yet reviewed" —
there is no placeholder-row overhead. No VLM calls, no per-patent JSON/HTML,
no external API calls, and no copying of approved files into `reviewed/`
(that finalize step stays in `01_review.ipynb`).

SigLIP and PatentSBERTa are **required** — the notebook will not run without
them. `src/reviewer.py`, `src/cross_modal.py`, `src/excel_schema.py` and
`01_review.ipynb` are all read but never modified by this notebook.

In [1]:
# ── GPU selection + VRAM check (same pattern as 01_review.ipynb) ───────────
# GPU_SELECTION:
#   "auto"    -> use every GPU with enough free VRAM (>= MIN_FREE_GB)
#   "0"       -> force a single worker on GPU 0 only
#   "1"       -> force a single worker on GPU 1 only
#   "0,1"     -> force both GPUs regardless of current free memory
# When 2+ GPUs end up selected, Cell 4 (T1 pre-labels) splits patent_ids
# across one worker subprocess per GPU — same pattern as 01_review.ipynb's
# run_stage01_parallel()/review_gpu_worker.py. Duplicate detection (Cell 5)
# always runs single-GPU afterward on selected_gpu_ids[0] — 01_review.ipynb
# doesn't parallelize that step either (it's a single pairwise comparison
# over the whole batch, not an independent per-patent job).
GPU_SELECTION = "0,1"
MIN_FREE_GB   = 4.0   # PatentSBERTa + SigLIP need far less headroom than the full 00b2/Qwen pipeline

import torch

def _free_gb(idx: int) -> float:
    free_bytes, _total_bytes = torch.cuda.mem_get_info(idx)
    return free_bytes / 1e9

n_gpu = torch.cuda.device_count()
print(f"GPUs visible: {n_gpu}")
free_per_gpu = {}
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    free = _free_gb(i)
    free_per_gpu[i] = free
    flag = "OK" if free >= MIN_FREE_GB else "LOW"
    print(f"  GPU {i} ({name}): {free:.1f} GB free  [{flag}]")

if GPU_SELECTION == "auto":
    selected_gpu_ids = [str(i) for i in range(n_gpu) if free_per_gpu.get(i, 0) >= MIN_FREE_GB]
    if not selected_gpu_ids:
        best = max(free_per_gpu, key=free_per_gpu.get, default=0)
        selected_gpu_ids = [str(best)]
        print(f"\nNo GPU has >= {MIN_FREE_GB} GB free -- falling back to GPU {best} only "
              f"({free_per_gpu.get(best, 0):.1f} GB free).")
    else:
        print(f"\nAuto-selected GPU(s): {', '.join(selected_gpu_ids)}")
else:
    selected_gpu_ids = [s.strip() for s in GPU_SELECTION.split(",") if s.strip()]
    print(f"\nForced GPU selection: {', '.join(selected_gpu_ids)}")
    for s in selected_gpu_ids:
        idx = int(s)
        free = free_per_gpu.get(idx, 0)
        if free < MIN_FREE_GB:
            print(f"  WARNING: GPU {idx} only has {free:.1f} GB free (< {MIN_FREE_GB} GB) -- "
                  f"this worker may hit OOM.")

print(f"\nT1 pre-label workers: {len(selected_gpu_ids)} -> GPU(s) {selected_gpu_ids}")

GPUs visible: 2
  GPU 0 (NVIDIA GeForce RTX 2080 Ti): 10.6 GB free  [OK]
  GPU 1 (NVIDIA GeForce RTX 2080 Ti): 11.0 GB free  [OK]

Forced GPU selection: 0, 1

T1 pre-label workers: 2 -> GPU(s) ['0', '1']


In [2]:
# ── Repo root on sys.path (same bootstrap as 01_review.ipynb) ──────────────
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")

# ── Config + batch selector (same BATCH_ID pattern as 01_review.ipynb) ─────
BATCH_ID = 3   # see data/batches.xlsx Summary sheet — Batch_01 .. Batch_05

# Set to a folder name under matched/ to bypass batches.xlsx entirely (e.g. a
# tester batch with no batches.xlsx sheet). Leave as None for a real run.
OVERRIDE_BATCH_DIR = None   # e.g. "Batch_01_review_tester"

import pandas as pd
from src.config_loader import load_config

cfg = load_config()

if OVERRIDE_BATCH_DIR:
    sheet_name = OVERRIDE_BATCH_DIR
    matched_probe = Path(cfg["paths"]["matched"]) / sheet_name
    patent_ids = sorted(p.name for p in matched_probe.iterdir() if p.is_dir())
    print(f"OVERRIDE: {len(patent_ids)} patents loaded from matched/{sheet_name}/ (no batches.xlsx sheet used)")
else:
    batches_path = cfg["paths"]["batches_xlsx"]
    if not batches_path.exists():
        raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

    sheet_name  = f"Batch_{BATCH_ID:02d}"
    batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
    patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

    print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")

# Cap the number of patents processed — handy for a quick test run before
# committing to the whole batch. Same LIMIT pattern as 01_review.ipynb's
# "Batch run configuration" cell. None = process all patents in this batch.
LIMIT = None   # e.g. 5 to only run the first 5 patents
if LIMIT:
    patent_ids = patent_ids[:LIMIT]
    print(f"LIMIT={LIMIT} — processing only the first {len(patent_ids)} patent(s).")

matched_dir = Path(cfg["paths"]["matched"]) / sheet_name
print(f"matched_dir : {matched_dir}")

# PatSeer metadata (title/abstract/assignee/pub_year/app_year/citations) —
# same loader 01_review.ipynb uses, needed for T1 metadata rows + T1 SBERT
# pre-labels + text-duplicate detection below.
from src.extractor import load_patseer_excel
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
print(f"Indexed {len(excel_index)} patents from PatSeer excel.")

repo_root : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools
Batch 3: 400 patents loaded from Batch_03
matched_dir : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/00b2_figure_crops/Batch_03


/home/vasco/anaconda3/envs/nb_01_review/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [3]:
# ── Model loading: PatentSBERTa + SigLIP only ───────────────────────────────
# SigLIP loaded for image duplicate detection only — not used for field
# classification (the full T2/G1/M1/M2/M3 SigLIP zero-shot pass stays in
# 01_review.ipynb's run_stage01()/process_patent(), which this notebook never
# calls).
import os

# Must run BEFORE any sentence_transformers/open_clip/huggingface_hub import:
# huggingface_hub freezes HF_HUB_CACHE as a constant at first import, so
# setting it afterward is silently ignored and weights re-download to the
# default ~/.cache/huggingface instead of the project models/ folder.
os.environ["HF_HUB_CACHE"] = str(cfg["paths"]["siglip_cache"])
os.environ["HF_HOME"]      = str(cfg["paths"]["siglip_cache"])

from src.cross_modal import load_siglip_model

# Use whichever GPU the "GPU selection + VRAM check" cell above picked
# (selected_gpu_ids) instead of the bare "cuda" string, which always means
# device 0 regardless of GPU_SELECTION.
_selected = globals().get("selected_gpu_ids")
model_device = f"cuda:{_selected[0]}" if _selected else None

# -- PatentSBERTa -- required for T1 dimension classification --------------
# Cached under cfg["paths"]["sbert_cache"] (Patent-Labelling-Tools/models/SBERT)
# so weights stay inside the project instead of ~/.cache/huggingface.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"]), device=model_device)
print(f"PatentSBERTa ready on {sbert.device}.")

# -- SigLIP -- required for image duplicate detection only ------------------
siglip_bundle = load_siglip_model(cache_dir=cfg["paths"]["siglip_cache"], device=model_device)
print(f"SigLIP ready on {siglip_bundle[3]}.")

I0000 00:00:1786989337.878702  158190 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


PatentSBERTa ready on cuda:0.
[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda:0
SigLIP ready on cuda:0.


In [4]:
# ── T1 SBERT pre-labels (scope / t1Field / t1Target) ────────────────────────
# classify_t1_dimensions()'s own docstring defines its input text as
# "title + abstract + description of drawings" — that's exactly what's built
# here. This is narrower than process_patent()'s fuller classify_text (which
# also folds in first_claim + innovation_objective) since this notebook never
# calls process_patent() at all — see the module docstring / summary cell for
# why that's a deliberate scope difference, not an oversight.
#
# GPU split: when Cell 1 selected 2+ GPUs, this spawns one worker subprocess
# per GPU (src/wizard_feed_gpu_worker.py) — same pattern as 01_review.ipynb's
# run_stage01_parallel()/review_gpu_worker.py, scoped down to just T1
# classification (the only thing this notebook predicts). Each worker's
# progress is streamed with a "[GPU X]" prefix so you can see which patent
# finished on which card. A single selected GPU just runs in-process instead
# (no subprocess overhead) — same threshold 01_review uses.
import json as _json
import os
import subprocess
import sys
import tempfile
import threading
import shutil
import pandas as pd
from pathlib import Path

_desc_csv = Path(cfg["paths"]["data"]) / "descriptions.csv"
if _desc_csv.exists():
    _desc_df = pd.read_csv(_desc_csv, dtype=str).fillna("")
    desc_by_patent = dict(zip(_desc_df["patent_id"], _desc_df["description_of_drawings"]))
else:
    desc_by_patent = {}
    print(f"⚠ {_desc_csv} not found — T1 pre-labels will run on title+abstract only.")

texts_by_patent = {}
for pid in patent_ids:
    excel_row = excel_index.get(pid, {})
    texts_by_patent[pid] = " ".join(t for t in [
        excel_row.get("title"), excel_row.get("abstract"), desc_by_patent.get(pid, ""),
    ] if t)

t1_predictions = {}

if len(selected_gpu_ids) > 1:
    print(f"Splitting T1 pre-labels across {len(selected_gpu_ids)} GPU(s): {selected_gpu_ids}")

    n_workers = len(selected_gpu_ids)
    chunk  = max(1, -(-len(patent_ids) // n_workers))  # ceil division
    splits = [patent_ids[i:i + chunk] for i in range(0, len(patent_ids), chunk)] or [[]]
    while len(splits) < n_workers:
        splits.append([])

    worker_script = str(Path(repo_root) / "src" / "wizard_feed_gpu_worker.py")
    python_exe    = sys.executable

    tmp_dir = Path(tempfile.mkdtemp(prefix="wizard_feed_parallel_"))
    procs, result_paths = [], []

    for i in range(n_workers):
        if not splits[i]:
            result_paths.append(None)
            continue
        args_path   = tmp_dir / f"args_{i}.json"
        result_path = tmp_dir / f"result_{i}.json"
        result_paths.append(result_path)

        args_path.write_text(_json.dumps({
            "patent_ids": splits[i],
            "texts":      {pid: texts_by_patent[pid] for pid in splits[i]},
        }))

        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = selected_gpu_ids[i]

        p = subprocess.Popen(
            [python_exe, worker_script, str(args_path), str(result_path)],
            cwd=str(repo_root), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        procs.append(p)
        print(f"[GPU {selected_gpu_ids[i]}] worker started (PID {p.pid}) — {len(splits[i])} patent(s)")

    def _stream(proc, label):
        for line in proc.stdout:
            print(f"[GPU {label}] {line}", end="", flush=True)

    threads = [threading.Thread(target=_stream, args=(procs[j], selected_gpu_ids[j]), daemon=True)
               for j in range(len(procs))]
    for t in threads: t.start()
    try:
        for p in procs:   p.wait()
        for t in threads: t.join()
    except BaseException:
        for p in procs:
            if p.poll() is None:
                p.terminate()
        for p in procs:
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                p.kill()
        raise

    for rp in result_paths:
        if rp is None:
            continue
        if not rp.exists() or rp.stat().st_size == 0:
            print(f"⚠  Result file missing/empty: {rp} — that worker crashed before writing "
                  f"results (its patents are NOT in this run's output — check the [GPU ...] "
                  f"traceback above, e.g. a CUDA OOM, and re-run once fixed).")
            continue
        try:
            data = _json.loads(rp.read_text())
        except _json.JSONDecodeError:
            print(f"⚠  Result file corrupt: {rp} — that worker crashed mid-write; skipping it.")
            continue
        t1_predictions.update(data["t1_predictions"])

    shutil.rmtree(tmp_dir, ignore_errors=True)

else:
    from tqdm import tqdm
    from src.reviewer import classify_t1_dimensions
    for pid in tqdm(patent_ids, desc="T1 SBERT pre-labels"):
        t1_predictions[pid] = classify_t1_dimensions(texts_by_patent[pid], sbert)

n_predicted = sum(
    1 for pid in patent_ids
    if any((t1_predictions.get(pid) or {}).get(f, {}).get("value") for f in ("scope", "t1Field", "t1Target"))
)
print(f"T1 pre-labels computed for {len(t1_predictions)} patent(s) ({n_predicted} with at least one non-empty field).")

⚠ /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/descriptions.csv not found — T1 pre-labels will run on title+abstract only.
Splitting T1 pre-labels across 2 GPU(s): ['0', '1']
[GPU 0] worker started (PID 158983) — 200 patent(s)
[GPU 1] worker started (PID 158984) — 200 patent(s)
[GPU 0] WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
[GPU 0] I0000 00:00:1786989356.187444  158983 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
[GPU 0] To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[GPU 1] WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
[GPU 1] I0000 00:00:1786989356.334910  158984 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical op

In [5]:
# ── Duplicate detection (SigLIP image-to-image, PatSeer family ID, PatentSBERTa text-to-text) ─
# Reuses detect_image_duplicates()/detect_text_duplicates() from
# src/cross_modal.py EXACTLY as 01_review.ipynb's duplicate-detection cell
# does — same functions, same call signature. The only difference is WHERE
# the input comes from: 01_review reads image entries back from an already-
# written ml_predict_labels_<batch>.xlsx (populated earlier by run_stage01's
# full T2 pass); this notebook never writes T2 classification rows, so
# entries are built directly from the figure crops on disk instead (same
# glob pattern process_patent() itself uses to discover a patent's crops).
#
# Thresholds recalibrated against a Batch_01 dry run that used the
# 01_review.ipynb defaults (0.85 / 0.70): those flagged 88% of all figure
# crops and 62% of all patents as duplicates. Patent line-art embeds in a
# very tight SigLIP cluster (every figure is a B/W aircraft drawing) and this
# whole corpus is a single tech domain (eVTOLs) in SBERT space, so 0.85/0.70
# were measuring "another aircraft drawing" / "another eVTOL patent", not
# "the same drawing" / "the same invention". Both are re-exported as
# SUGGESTIONS the reviewer must explicitly confirm (see the HTML wizard's
# MACHINE_SOURCES gate — an ML dup row no longer auto-commits), so the cost
# of a slightly-too-high threshold here is just a missed suggestion, not a
# silently wrong export; a too-low threshold was the real hazard (mass
# false positives auto-copying labels across unrelated figures/patents).
IMAGE_DUPLICATE_THRESHOLD = 0.97   # was 0.85
TEXT_MATCH_THRESHOLD      = 0.93   # was 0.70 — now a SECONDARY/fallback signal, see below

import re
from pathlib import Path
from src.cross_modal import detect_image_duplicates, detect_text_duplicates
from src.reviewer import resolve_patent_image_dir

# fig_num parsed from 'filename', same convention as the HTML wizard's
# figNumFromFilename() and 01_review.ipynb's duplicate-detection cell.
def _fig_num(fname):
    m = re.search(r'_Fu(\d+)\.png$', fname)
    if m: return 'Fu' + m.group(1)
    if re.search(r'_Fu\.png$', fname):
        d = re.search(r'_D(\d+)_', fname)
        return ('Fu_D' + d.group(1)) if d else fname.rsplit('.', 1)[0]
    m = re.search(r'_F(\w+)\.png$', fname)
    if m: return m.group(1)
    return fname.rsplit('.', 1)[0]

# Both detect_*_duplicates() functions treat "earlier in the input" as the
# canonical original (see their docstrings). patent_ids comes from
# batches.xlsx in arbitrary sheet order, so without this, dup_of_patent could
# point at a LATER-filed patent. Sort earliest-publication-first instead;
# patents with no pub_year sort last (never chosen as an "original" over a
# dated one).
def _pub_year_key(pid):
    y = (excel_index.get(pid, {}) or {}).get("pub_year")
    return y if y else "9999"
ordered_patent_ids = sorted(patent_ids, key=_pub_year_key)

# -- Collect one image entry per figure crop (same glob pattern process_patent() uses) --
_entries = []
for pid in ordered_patent_ids:
    patent_img_dir = resolve_patent_image_dir(matched_dir, pid)
    if not patent_img_dir.exists():
        continue
    labeled   = sorted(patent_img_dir.glob("*_F[0-9]*.png"))
    unlabeled = sorted(patent_img_dir.glob("*_Fu*.png"))
    for f in labeled + unlabeled:
        _entries.append({"patent_id": pid, "fig_num": _fig_num(f.name),
                          "filename": f.name, "image_path": str(f)})

print(f"Encoding {len(_entries)} figure crop(s) for image-duplicate detection...")
_model, _tok, _pre, _dev = siglip_bundle
image_duplicates = detect_image_duplicates(_entries, _model, _pre, _dev,
                                            threshold=IMAGE_DUPLICATE_THRESHOLD)
print(f"  {len(image_duplicates)} duplicate image(s) found (threshold {IMAGE_DUPLICATE_THRESHOLD}).")

# -- Simple Family ID (PatSeer) -- PRIMARY, exact patent-duplicate signal --
# Free and precise: two Record Numbers sharing a Simple Family ID are the same
# underlying invention (continuations/national-phase filings/etc.), which is
# exactly what "duplicate patent" means here — far more reliable than text-
# embedding similarity, which can't distinguish "same invention" from "same
# tech domain". Read directly off the PatSeer sheet since load_patseer_excel()
# doesn't expose this column.
import pandas as pd
_family_df = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str,
                            usecols=["Record Number", "Simple Family ID"])
family_by_patent = dict(zip(_family_df["Record Number"].str.strip(),
                             _family_df["Simple Family ID"].fillna("").str.strip()))

family_duplicates = {}   # dup patent_id -> {"dup_of_patent": str, "family_id": str}
_first_by_family = {}
for pid in ordered_patent_ids:
    fam = family_by_patent.get(pid, "")
    if not fam:
        continue
    if fam in _first_by_family:
        family_duplicates[pid] = {"dup_of_patent": _first_by_family[fam], "family_id": fam}
    else:
        _first_by_family[fam] = pid
print(f"  {len(family_duplicates)} family-duplicate patent(s) found (Simple Family ID).")

# -- PatentSBERTa text-to-text -- SECONDARY suggestion, title+abstract only --
# Only run on patents family-ID did NOT already flag — family membership is
# the trusted signal, SBERT is a lower-confidence "might be worth checking"
# flag layered on top, not a second vote on the same patents.
text_by_patent = {}
for pid in ordered_patent_ids:
    if pid in family_duplicates:
        continue
    excel_row = excel_index.get(pid, {})
    title, abstract = excel_row.get("title") or "", excel_row.get("abstract") or ""
    text_by_patent[pid] = (title + ". " + abstract).strip(". ")

text_duplicates = detect_text_duplicates(text_by_patent, sbert, threshold=TEXT_MATCH_THRESHOLD)
print(f"  {len(text_duplicates)} additional text-duplicate patent(s) found (SBERT threshold {TEXT_MATCH_THRESHOLD}).")

# Combined view used by the export cell — family (primary) entries win over an
# SBERT entry for the same patent_id (can't happen given the filter above, but
# family is authoritative either way). Source records provenance instead of a
# guessed duplicateType — see Cell 6 for why duplicateType is left for the
# reviewer to pick rather than inferred here.
all_text_duplicates = {}
for pid, info in family_duplicates.items():
    all_text_duplicates[pid] = {"dup_of_patent": info["dup_of_patent"], "source": "patseer_family"}
for pid, info in text_duplicates.items():
    all_text_duplicates[pid] = {"dup_of_patent": info["dup_of_patent"], "source": "sbert"}
print(f"  {len(all_text_duplicates)} text-duplicate patent(s) total.")


Encoding 2502 figure crop(s) for image-duplicate detection...
  449 duplicate image(s) found (threshold 0.97).


/home/vasco/anaconda3/envs/nb_01_review/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  0 family-duplicate patent(s) found (Simple Family ID).
  32 additional text-duplicate patent(s) found (SBERT threshold 0.93).
  32 text-duplicate patent(s) total.


In [6]:
# ── Export ml_predict_labels_{batch}.xlsx ───────────────────────────────────
# Same Review-sheet schema run_stage01() writes (src/excel_schema.py) — reuses
# its _row()/export_source_excel() so a row written here is byte-for-byte the
# same shape the wizard's rowsToAIData() already knows how to read. Only T1
# metadata + T1 pre-labels + duplicate flags are written; every T2/G1/M1/M2/M3
# classification field is simply absent (no empty/placeholder rows — the HTML
# wizard's fresh() state already treats a missing field as "not yet reviewed").
from pathlib import Path
from src.excel_schema import _row, export_source_excel

# T1 auto-approval heuristic — same logic build_patent_rows() applies (see its
# _AUTO_APPROVAL_CONF_THRESHOLD comment): a SUGGESTION, not a final call, so
# Needs_Review stays True either way and the human always confirms/overrides
# it in the wizard rather than silently trusting the heuristic. (The wizard
# itself gates isHumanReviewed on this row's Source being non-machine, so an
# auto-approved patent still can't masquerade as reviewed until a human
# actually clicks Approve/Disapprove.)
#   - all three of scope/t1Field/t1Target predicted, avg confidence >= 0.45
#     -> isApproved = True
#   - all three came back completely empty -> isApproved = False,
#     t1DisapproveReason = "Unreadable"
#   - anything in between (partial/low-confidence) -> neither row is written,
#     same as build_patent_rows() leaving them blank in that case.
_AUTO_APPROVAL_CONF_THRESHOLD = 0.45

rows = []
for pid in patent_ids:
    excel_row = excel_index.get(pid, {})

    # T1 metadata — always written (needed for the T1 approve/disapprove
    # triage step itself); same fields/definitions as build_patent_rows().
    for field, definition in [
        ("title", "Patent Title"), ("abstract", "Abstract"),
        ("assignee", "Assignee"), ("pub_year", "Publication Year"),
        ("app_year", "Application Year"),
    ]:
        rows.append(_row(pid, "T1", definition, field, definition, "", excel_row.get(field)))
    rows.append(_row(pid, "T1", "Description of Drawings", "description_of_drawings",
                      "Description of Drawings", "", desc_by_patent.get(pid)))

    # T1 pre-labels (scope / t1Field / t1Target). Source is forced to "sbert"
    # per this notebook's spec, overriding classify_t1_dimensions()'s own
    # source="auto" tag (see Cell 4 / summary cell) — every OTHER prediction
    # source this schema uses is a fixed literal ("siglip", "auto_heuristic",
    # "ensemble", ...) rather than the modality name, so "sbert" here just
    # follows the same convention while being unambiguous about provenance.
    preds = t1_predictions.get(pid, {})
    scope_p, field_p, target_p = preds.get("scope") or {}, preds.get("t1Field") or {}, preds.get("t1Target") or {}
    for field, pred in (("scope", scope_p), ("t1Field", field_p), ("t1Target", target_p)):
        value = pred.get("value")
        rows.append(_row(pid, "T1", field, field, field, "",
                          value, pred.get("confidence"), "sbert" if value else None))

    scope_v, field_v, target_v = scope_p.get("value"), field_p.get("value"), target_p.get("value")
    _confs = [c for c in (scope_p.get("confidence"), field_p.get("confidence"), target_p.get("confidence"))
              if isinstance(c, (int, float))]
    avg_conf = sum(_confs) / len(_confs) if _confs else None

    auto_approved, auto_reason = None, None
    if scope_v and field_v and target_v and avg_conf is not None and avg_conf >= _AUTO_APPROVAL_CONF_THRESHOLD:
        auto_approved = True
    elif not scope_v and not field_v and not target_v:
        auto_approved, auto_reason = False, "Unreadable"

    if auto_approved is not None:
        rows.append(_row(pid, "T1", "T1 — Approval Status", "isApproved", "T1 — Approval Status",
                          "true|false", auto_approved, avg_conf, "auto_heuristic", needs_review=True))
    if auto_reason is not None:
        rows.append(_row(pid, "T1", "T1 — Disapproval Reason", "t1DisapproveReason",
                          "T1 — Disapproval Reason", "Pure UAV|Out of Domain|Unreadable|No Aircraft Image|Other",
                          auto_reason, None, "auto_heuristic", needs_review=True))

# Duplicate-image rows (dupOfPatent / dupOfFig) — one pair per flagged figure,
# same Value/Source shape as 01_review.ipynb's duplicate-detection cell writes
# (no Confidence column — 01_review doesn't set one for these rows either).
# Dict keyed by (patent_id, fig_num) instead of a linear scan of _entries per
# duplicate (that was O(len(image_duplicates) * len(_entries)) — ~11M
# comparisons on a 352-patent/3546-crop batch; this is O(1) per lookup).
_entry_by_key = {(e["patent_id"], e["fig_num"]): e for e in _entries}
for (pid, fig_num), info in image_duplicates.items():
    entry = _entry_by_key.get((pid, fig_num))
    if entry is None:
        continue
    sub_dim, img_path = f"Image: {entry['filename']}", entry["image_path"]
    rows.append(_row(pid, "T2", sub_dim, "dupOfPatent", f"{sub_dim} — duplicate of patent", "",
                      info["dup_of_patent"], None, "siglip", img_path))
    rows.append(_row(pid, "T2", sub_dim, "dupOfFig", f"{sub_dim} — duplicate of figure", "",
                      info["dup_of_fig"], None, "siglip", img_path))

# Duplicate-text rows (isDuplicate / duplicateId) — same Needs_Review=True
# pattern that 01_review.ipynb's _set_t1() helper writes. Source records
# provenance ("patseer_family" = exact simple-family match, "sbert" = embedding
# suggestion — see Cell 5). No duplicateType row is written: unlike the family/
# text signals above, "Same Aircraft" vs. "Continuation" vs. "Component
# Overlap" (DUP_TYPES in the wizard) is a judgment call neither signal can
# safely make, and getting it wrong isn't harmless — duplicateType='1' (Same
# Aircraft) is what the wizard's "Copy M1–M3" button keys off. Left for the
# reviewer to pick after looking at both patents.
for pid, info in all_text_duplicates.items():
    rows.append(_row(pid, "T1", "T1 — Duplicate Flag", "isDuplicate",
                      "T1 — Duplicate Flag", "true|false", True, None, info["source"], needs_review=True))
    rows.append(_row(pid, "T1", "T1 — Duplicate Of (Patent ID)", "duplicateId",
                      "T1 — Duplicate Of (Patent ID)", "", info["dup_of_patent"], None, info["source"], needs_review=True))

batch_label  = sheet_name
data_matched = Path(cfg["paths"].get("data_matched", cfg["paths"]["data"]))
out_path     = data_matched / batch_label / f"ml_predict_labels_{batch_label}.xlsx"

export_source_excel(rows, out_path)
print(f"Wrote {len(rows)} row(s) -> {out_path}")

# ── Post-export enrichment (same scripts 01_review.ipynb runs after export) ─
# Both scripts back up the xlsx before writing (timestamped) and are
# idempotent (strip prior rows first), reused unchanged.
import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 1) PDF link — the wizard's "Open PDF" button reads T1/pdf_link
# (metaVal('pdf_link')) and falls back to the Espacenet/EPO link when that
# row is missing. The PatSeer PDF URL lives ONLY as an Excel hyperlink on the
# source PatSeer sheet (not in excel_index), so it has to be injected here.
from scripts.inject_pdf_links import load_patseer_pdf_links, inject as _inject_pdf_links

_pdf_links = load_patseer_pdf_links(Path(cfg["paths"]["patseer_excel"]))
print(f"\n1) PatSeer PDF links available: {len(_pdf_links)}")
_inject_pdf_links(out_path, _pdf_links)

# 2) Missing-from-download audit — flags patents with no raw/ download folder
# (their images are placeholders only) so the wizard shows the red banner
# instead of silently letting them look like a normal, fully-downloaded patent.
from scripts.flag_missing_downloads import downloaded_publication_numbers, flag as _flag_missing_downloads

_downloaded = downloaded_publication_numbers(Path(cfg["paths"]["raw_images"]))
print(f"\n2) Download folders found: {len(_downloaded)}")
_n_missing = _flag_missing_downloads(out_path, _downloaded)

# 3) Disk-only images — this notebook writes NO T2 image rows at all except
# duplicate-flag rows on figures SigLIP happened to flag (Cell 5 above), so
# without this step the wizard's figure grid would be empty for almost every
# patent. Gives every image file in matched/<batch>/<patent>/ a T2
# "Image: <fname>" row (Image_Path filled) so it shows up for the human to
# label from scratch — including crops 00b2 produced but this notebook never
# classifies, and any file the reviewer manually drops into the folder later
# (see the HTML wizard's 📎 Attach Image button).
from scripts.inject_missing_images import inject as _inject_disk_imgs

print("\n3) Disk-only images (every file in matched/<batch>/<patent>/ not already a row)")
_inject_disk_imgs(out_path, Path(cfg["paths"]["matched"]), batch_label, inject_all=False)


Wrote 4735 row(s) -> /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/00b2_crops_and_01a_MACHINE_feed/Batch_03/ml_predict_labels_Batch_03.xlsx


/home/vasco/anaconda3/envs/nb_01_review/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



1) PatSeer PDF links available: 1628
  removed 0 stale pdf_link row(s)
  patents in export      : 400
  pdf_link rows added    : 398
  patents with NO PatSeer link: 2
  backup                 : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/00b2_crops_and_01a_MACHINE_feed/Batch_03/ml_predict_labels_Batch_03.PRE_PDFLINK_20260817_190529.xlsx
  written                : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/00b2_crops_and_01a_MACHINE_feed/Batch_03/ml_predict_labels_Batch_03.xlsx

2) Download folders found: 1614
  removed 0 stale missing_from_download row(s)
  patents in export          : 400
  MISSING FROM DOWNLOAD      : 1
      - GB201817002D0
  backup                     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/00b2_crops_and_01a_MACHINE_feed/Batch_03/ml_predict_labels_Batch_03.PRE_MISSINGDL_20260817_190531.xlsx
  written

In [7]:
# ── Summary ──────────────────────────────────────────────────────────────
n_t1_written = sum(
    1 for pid in patent_ids
    if any((t1_predictions.get(pid) or {}).get(f, {}).get("value") for f in ("scope", "t1Field", "t1Target"))
)

print("=" * 55)
print(f"  01a_wizard_feed complete: {sheet_name}")
print(f"  Patents processed          : {len(patent_ids)}")
print(f"  T1 pre-labels written      : {n_t1_written}")
print(f"  Duplicate images flagged   : {len(image_duplicates)}")
print(f"  Duplicate patents (family) : {len(family_duplicates)}")
print(f"  Duplicate patents (sbert)  : {len(text_duplicates)}")
print(f"  Duplicate patents (total)  : {len(all_text_duplicates)}")
print(f"  Missing from download      : {_n_missing}")
print(f"  Output                     : {out_path}")
print("=" * 55)

  01a_wizard_feed complete: Batch_03
  Patents processed          : 400
  T1 pre-labels written      : 400
  Duplicate images flagged   : 449
  Duplicate patents (family) : 0
  Duplicate patents (sbert)  : 32
  Duplicate patents (total)  : 32
  Missing from download      : 1
  Output                     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/00b2_crops_and_01a_MACHINE_feed/Batch_03/ml_predict_labels_Batch_03.xlsx
